# Evaluación múltiple — métricas promediadas sobre los 20 clips

Lee los JSON reducidos (`integrado_eval_SNMOT-XXX.json`) generados por `07_pipeline_reducido_lote.ipynb` y evalúa, sobre los 20 clips de test de SoccerNet, cuatro aspectos del sistema:

- **Detección** — ¿está cada objeto en el sitio correcto? (precision/recall/F1 de persona, árbitro y balón)
- **Tracking** — ¿se mantiene la identidad de cada jugador a lo largo del tiempo? (MOTA, IDF1, ID switches)
- **Clasificación de equipos** — ¿se asigna bien cada jugador a su equipo? (acierto + matriz de confusión)
- **Diagnósticos** — desglose del árbitro (detección vs etiquetado) y del balón (crudo vs interpolado)

---

## Cómo leer las métricas (bloques comunes)

Casi todo se construye emparejando **tus cajas** con las del **ground truth (GT)** frame a frame. El emparejamiento es por **IoU ≥ 0.5** para personas y árbitro, y por **distancia de centros ≤ 15 px** para el balón (la IoU es inestable en cajas diminutas). De cada emparejamiento salen tres conteos:

- **TP** (verdadero positivo): caja tuya que casa con una del GT → acierto.
- **FP** (falso positivo): caja tuya sin correspondencia → detección de más (duplicado, fondo confundido).
- **FN** (falso negativo): caja del GT que nadie cubre → objeto perdido.

Y de ahí las tres métricas base:

- **Precision = TP / (TP + FP)** — *pureza*. Qué fracción de lo que detectas es real. La bajan los falsos positivos. Una precisión baja ensucia el tablero con objetos fantasma.
- **Recall = TP / (TP + FN)** — *cobertura*. Qué fracción de lo que realmente había capturas. Lo baja todo lo que el detector no ve (oclusión, escala pequeña, desenfoque). Un recall bajo deja huecos que luego fragmentan el tracking.
- **F1 = media armónica de P y R** — resumen en un número; penaliza si una de las dos es baja. Útil para comparar, pero **esconde de qué lado falla**: hay que mirar siempre P y R por separado.

Dos formas de agregar los 20 clips, y la diferencia importa:

- **Micro-promedio**: se suman TP/FP/FN de los 20 clips y se calcula P/R una sola vez. Pondera por número de objetos → los clips más poblados pesan más. Es la **cifra global "de titular"**.
- **Media ± std por clip**: se calcula P/R en cada clip y se promedia. Cada clip pesa igual; la **std mide robustez**. Una std alta avisa de clips problemáticos aunque el micro se vea bien.

---

## 1. Detección — ¿está el objeto en el sitio? (ignora identidad)

Mide solo *localización*, no *identidad*. Se evalúan tres clases por separado, porque su dificultad es muy distinta:

- **Persona** (jugadores + porteros): es la base de todo el sistema. Su **recall** es la métrica que limita aguas abajo: lo que no se detecta no se puede trackear, clasificar ni analizar.
- **Árbitro**: clase minoritaria; suele tener más varianza entre clips.
- **Balón**: clase diminuta y muy intermitente (desenfoque, oclusión, lejanía); su recall en crudo es típicamente el más bajo del sistema.

**En qué influye**: el umbral de confianza del detector mueve el equilibrio P↔R (bajarlo sube recall y falsos positivos; subirlo, al revés). La calidad del entrenamiento por clase explica las diferencias entre persona/árbitro/balón.

---

## 2. Tracking — ¿se mantiene la identidad en el tiempo?

Aquí ya no basta con detectar: importa que el **ID** de cada jugador sea estable frame a frame. Se usa la librería estándar `motmetrics`, **solo sobre personas** (balón y árbitro quedan fuera), emparejando por IoU ≥ 0.5.

- **MOTA** (*Multiple Object Tracking Accuracy*) = 1 − (FN + FP + IDSW) / total_GT. Mete omisiones, falsos positivos y cambios de identidad en un único número.
  - *En qué influye*: está **dominada por FN + FP**, que casi siempre superan con mucho a los switches. En la práctica es "**calidad de detección con un castigo pequeño por identidad**". Buena para la salud global, pero **no es un buen indicador de identidad por sí sola**.

- **IDF1** (*ID F1-score*) = F1 de las detecciones **correctamente identificadas** bajo la asignación óptima de IDs en toda la secuencia.
  - *En qué influye*: muy **sensible a la fragmentación y a los intercambios de identidad**. Penaliza partir un mismo jugador en varios IDs o cruzar dos IDs. Lo suben el re-ID por apariencia y un buen modelo de movimiento; lo bajan oclusiones, cruces y camisetas iguales.
  - *Clave*: la **brecha entre MOTA y IDF1** es la firma del tracking. MOTA alta + IDF1 baja = "detecto bien pero las identidades no aguantan".

- **ID switches (IDSW)**: número de veces que el ID asignado a un objeto del GT cambia.
  - *En qué influye*: es la **causa directa** de un IDF1 bajo. Cada switch rompe una trayectoria, lo que invalida cualquier análisis individual (distancia recorrida, mapa de calor por jugador).

- **mostly_tracked (MT)** / **mostly_lost (ML)**: número de trayectos del GT seguidos ≥ 80 % / perdidos ≥ 80 % de su vida. Dan una visión **por-trayecto** (complementa a IDF1, que es por-detección).
- **num_unique_objects**: número de identidades reales en el GT; es el denominador de MT (p. ej. MT/únicos = fracción de jugadores "casi siempre" seguidos).

---

## 3. Clasificación de equipos

Evalúa el agrupador de color de camiseta (k-means → equipo 0/1). Hay un problema de fondo: **las etiquetas 0/1 son arbitrarias por clip** (el k-means no sabe cuál es "local/izquierda"). Por eso se usa **asignación óptima por clip**: en cada clip se empareja cada jugador detectado con su GT (IoU ≥ 0.5), se cuenta cómo se mapean tus clusters a los lados izquierda/derecha del GT (de `gameinfo.ini`) y se elige el **mejor de los dos mapeos** posibles. Esto elimina la ambigüedad de etiqueta y mide la **separación real**.

- **Acierto de equipo** = % de jugadores detectados-y-emparejados que reciben el equipo correcto.
  - *En qué influye*: la **separabilidad de los kits** (colores parecidos, sombras, iluminación, porteros con otra equipación lo bajan) y la **calidad de las cajas** (una caja mala da un crop de color malo).
  - *Importante*: está **condicionado a que el jugador se haya detectado**. Los jugadores perdidos por el detector no penalizan aquí (ya cuentan en el recall de detección). Es lo correcto para aislar el clasificador, pero hay que enunciarlo.
  - Se reporta en **micro** (pondera por nº de jugadores) y **por clip** (media / std / mínimo → consistencia y peor caso).

- **Matriz de confusión** (izq/der alineada por clip): reparto de aciertos y errores tras alinear cada clip al GT. **Errores simétricos** = sin sesgo sistemático; **asimétricos** = uno de los equipos se confunde más (típicamente un kit más ambiguo).

---

## 4. Diagnóstico de árbitro (complementario)

Para cada árbitro del GT busca la predicción que más solapa y la clasifica en tres cubos: **detectado y bien etiquetado**, **detectado pero etiquetado como jugador**, y **no detectado**.

- *Para qué sirve*: **separa el problema de detección del de rol**. Si domina "no detectado", hay que mejorar el **detector** (más datos de árbitro); si domina "→jugador", hay que mejorar la **lógica de asignación de rol**. Sin este desglose, un recall de árbitro bajo no te dice dónde actuar.

---

## 5. Balón: crudo vs interpolado (complementario)

Compara la salida cruda del balón con una versión interpolada que rellena huecos cortos (gate `MAX_GAP` en frames y `MAX_VEL` en px/frame, anti-teletransporte). Es la aproximación en píxeles del gate de 8 m de la fase 7 (aquí no hay campo).

- **Cobertura visual** = % de frames con una marca de balón (correcta o no). **No es precision ni recall**: mide la **continuidad** del tablero táctico (¿en cuántos frames hay balón que mostrar?).
- **P / R / F1 crudo vs interpolado**: el crudo es **preciso pero escaso**; la interpolación **sube recall y cobertura** a costa de **bajar precisión** (algunas posiciones interpoladas no casan con el GT, y cuando el GT no tiene balón anotado, el interpolado cuenta como FP). Es un **compromiso explícito continuidad ↔ precisión**, no un "mejor/peor".

---

## Qué métricas son las más importantes

Para un sistema de **análisis táctico** el orden de prioridad es:

1. **Recall y F1 de persona (detección)** — es el cimiento. Todo lo demás opera sobre los jugadores detectados; si el recall es bajo, ninguna métrica posterior puede recuperarse.
2. **IDF1 (tracking)** — la métrica de **identidad**, la que de verdad dice si el tracking sirve para análisis por jugador (trayectorias, distancias, mapas de calor individuales). Más informativa que MOTA, que es casi un proxy de detección.
3. **Acierto de equipo** — imprescindible para todo lo táctico a nivel de equipo (posesión, formaciones, ocupación de zonas).
4. **Recall / cobertura del balón** — necesario para posesión y pases; es el más difícil, así que se interpreta junto al compromiso crudo/interpolado.

El resto (precision, MOTA, ID switches, MT/ML, matriz de confusión, desglose de árbitro) son **importantes pero principalmente diagnósticas**: explican *por qué* suben o bajan las cuatro anteriores y dónde actuar para mejorarlas.

---

## Nota metodológica

Las métricas centrales se calculan sobre la **salida cruda** del pipeline, con **parámetros fijos** e idénticos para los 20 clips y **sin post-procesos selectivos** ni ajuste por clip — para que las cifras sean representativas y no estén infladas. El balón se evalúa por **distancia de centros** (no IoU). Las medidas de equipos y de árbitro están **condicionadas a la detección** (solo puntúan objetos previamente detectados y emparejados), por lo que deben leerse junto con el recall de detección, no de forma aislada. La comparación crudo/interpolado del balón es el **único** post-proceso, y se reporta de forma explícita para hacer visible su compromiso.

In [1]:
# ---- CELDA 1: cargar JSONs reducidos + ground truth (re-validación) ----
import subprocess, sys
subprocess.run([sys.executable,"-m","pip","install","motmetrics","--quiet"])

import json, csv, configparser
import numpy as np
from pathlib import Path
from collections import defaultdict, Counter

REPO_ROOT = Path("../..").resolve()
TEST_DIR = REPO_ROOT / "data" / "soccernet" / "tracking" / "test"
OUT_TRACKING = REPO_ROOT / "data" / "tracking_revalidacion"   # <-- LA MISMA carpeta donde escribió el notebook 07

# La lista de clips y el cargador usan AMBOS esta carpeta (no puede haber desajuste)
jsons = sorted(OUT_TRACKING.glob("integrado_eval_SNMOT-*.json"))
clips_eval = [j.stem.replace("integrado_eval_","") for j in jsons]
print(f"Carpeta de JSON: {OUT_TRACKING}")
print(f"Clips con JSON de evaluación: {len(clips_eval)}")
for c in clips_eval: print(f"  {c}")
assert len(clips_eval) > 0, f"No hay JSON en {OUT_TRACKING}. ¿El notebook 07 escribió en esta carpeta?"

def cargar_clip(nombre):
    with open(OUT_TRACKING / f"integrado_eval_{nombre}.json", encoding="utf-8") as fh:
        tu = json.load(fh)
    gt_por_frame = defaultdict(list)
    with open(TEST_DIR / nombre / "gt" / "gt.txt") as fh:
        for fila in csv.reader(fh):
            fr,tid=int(fila[0]),int(fila[1])
            x,y,w,h=float(fila[2]),float(fila[3]),float(fila[4]),float(fila[5])
            gt_por_frame[fr].append((tid,x,y,w,h))
    cfg=configparser.ConfigParser(); cfg.read(TEST_DIR/nombre/"gameinfo.ini")
    def cat(desc):
        d=desc.lower()
        if "ball" in d: return "balon"
        if "referee" in d: return "arbitro"
        if "player" in d or "goalkeeper" in d: return "persona"
        return "otro"
    clase_gt={}
    for key in cfg["Sequence"]:
        if key.startswith("trackletid_"): clase_gt[int(key.split("_")[1])]=cat(cfg["Sequence"][key])
    return tu, gt_por_frame, clase_gt

print("\nFunción de carga lista.")

Carpeta de JSON: C:\Users\sgghu\OneDrive\Documents\GitHub\TFG\data\tracking_revalidacion
Clips con JSON de evaluación: 29
  SNMOT-116
  SNMOT-117
  SNMOT-118
  SNMOT-119
  SNMOT-121
  SNMOT-122
  SNMOT-123
  SNMOT-124
  SNMOT-125
  SNMOT-126
  SNMOT-130
  SNMOT-133
  SNMOT-134
  SNMOT-136
  SNMOT-141
  SNMOT-143
  SNMOT-144
  SNMOT-145
  SNMOT-148
  SNMOT-187
  SNMOT-188
  SNMOT-190
  SNMOT-191
  SNMOT-192
  SNMOT-193
  SNMOT-195
  SNMOT-197
  SNMOT-199
  SNMOT-200

Función de carga lista.


In [2]:
# ---- CELDA 2: DETECCIÓN sobre los 20 clips (micro-promedio + por clip) ----
def iou_xywh(a,b):
    ax1,ay1,ax2,ay2=a[0],a[1],a[0]+a[2],a[1]+a[3]; bx1,by1,bx2,by2=b[0],b[1],b[0]+b[2],b[1]+b[3]
    ix1,iy1,ix2,iy2=max(ax1,bx1),max(ay1,by1),min(ax2,bx2),min(ay2,by2)
    if ix2<ix1 or iy2<iy1: return 0.0
    inter=(ix2-ix1)*(iy2-iy1); return inter/(a[2]*a[3]+b[2]*b[3]-inter)
def xywh(b): x1,y1,x2,y2=b; return (x1,y1,x2-x1,y2-y1)
def centro(b): return (b[0]+b[2]/2,b[1]+b[3]/2)
def cat_tuya(rol):
    if rol=="balon": return "balon"
    if rol=="arbitro": return "arbitro"
    if rol in ("jugador","portero"): return "persona"
    return "otro"

def eval_det_clip(tu, gt_por_frame, clase_gt, cats, usar_dist=False, IOU=0.5, DIST=15):
    TP=FP=FN=0
    for i,dets in enumerate(tu["frames"]):
        gtf=i+1
        gtb=[(x,y,w,h) for (tid,x,y,w,h) in gt_por_frame.get(gtf,[]) if clase_gt.get(tid) in cats]
        tub=[xywh(d["bbox"]) for d in dets if cat_tuya(d.get("rol")) in cats]
        usado=[False]*len(gtb)
        for tb in tub:
            mejor,mj=(DIST,-1) if usar_dist else (IOU,-1)
            for j,gb in enumerate(gtb):
                if usado[j]: continue
                if usar_dist:
                    ct,cg=centro(tb),centro(gb); d=np.hypot(ct[0]-cg[0],ct[1]-cg[1])
                    if d<mejor: mejor,mj=d,j
                else:
                    v=iou_xywh(tb,gb)
                    if v>mejor: mejor,mj=v,j
            if mj>=0: TP+=1; usado[mj]=True
            else: FP+=1
        FN+=usado.count(False)
    return TP,FP,FN

# Acumular sobre todos los clips
glob={c:{"TP":0,"FP":0,"FN":0} for c in ["persona","arbitro","balon"]}
por_clip={c:[] for c in ["persona","arbitro","balon"]}

for nombre in clips_eval:
    tu,gt,clase=cargar_clip(nombre)
    for cat,cats,dist in [("persona",{"persona"},False),("arbitro",{"arbitro"},False),("balon",{"balon"},True)]:
        TP,FP,FN=eval_det_clip(tu,gt,clase,cats,usar_dist=dist)
        glob[cat]["TP"]+=TP; glob[cat]["FP"]+=FP; glob[cat]["FN"]+=FN
        p=TP/(TP+FP) if TP+FP>0 else 0; r=TP/(TP+FN) if TP+FN>0 else 0
        por_clip[cat].append((p,r))

print("=== DETECCIÓN — MICRO-PROMEDIO (TP/FP/FN sumados de los 20 clips) ===\n")
for cat in ["persona","arbitro","balon"]:
    g=glob[cat]; TP,FP,FN=g["TP"],g["FP"],g["FN"]
    P=TP/(TP+FP) if TP+FP>0 else 0; R=TP/(TP+FN) if TP+FN>0 else 0
    F1=2*P*R/(P+R) if P+R>0 else 0
    print(f"{cat.upper():9}: P={P:.3f} R={R:.3f} F1={F1:.3f}  (TP={TP} FP={FP} FN={FN})")

print("\n=== DETECCIÓN — MEDIA ± STD por clip ===\n")
for cat in ["persona","arbitro","balon"]:
    ps=np.array([x[0] for x in por_clip[cat]]); rs=np.array([x[1] for x in por_clip[cat]])
    print(f"{cat.upper():9}: P={ps.mean():.3f}±{ps.std():.3f}  R={rs.mean():.3f}±{rs.std():.3f}")


=== DETECCIÓN — MICRO-PROMEDIO (TP/FP/FN sumados de los 20 clips) ===

PERSONA  : P=0.945 R=0.917 F1=0.931  (TP=258025 FP=14980 FN=23207)
ARBITRO  : P=0.865 R=0.898 F1=0.881  (TP=23372 FP=3657 FN=2653)
BALON    : P=0.811 R=0.439 F1=0.569  (TP=8959 FP=2093 FN=11463)

=== DETECCIÓN — MEDIA ± STD por clip ===

PERSONA  : P=0.944±0.018  R=0.914±0.037
ARBITRO  : P=0.864±0.093  R=0.895±0.061
BALON    : P=0.805±0.130  R=0.438±0.177


In [3]:
# (sanity check) recuento por clip — usa la MISMA carpeta que la CELDA 1 (sin reasignar OUT_TRACKING)
import json
from collections import Counter
for j in sorted(OUT_TRACKING.glob("integrado_eval_SNMOT-*.json")):
    tu = json.load(open(j, encoding="utf-8"))
    c = Counter(d.get("rol") for fr in tu["frames"] for d in fr)
    print(f"{j.stem.replace('integrado_eval_',''):11} n_frames={tu['n_frames']:4}  "
          f"jugador={c.get('jugador',0):5} arbitro={c.get('arbitro',0):4} "
          f"balon={c.get('balon',0):4}  total={sum(c.values())}")

SNMOT-116   n_frames= 750  jugador= 9138 arbitro= 624 balon= 151  total=9913
SNMOT-117   n_frames= 750  jugador= 7501 arbitro= 810 balon= 468  total=8779
SNMOT-118   n_frames= 750  jugador= 9206 arbitro= 940 balon= 506  total=10652
SNMOT-119   n_frames= 750  jugador= 6877 arbitro= 728 balon= 219  total=7824
SNMOT-121   n_frames= 750  jugador=10054 arbitro= 660 balon= 390  total=11104
SNMOT-122   n_frames= 750  jugador= 9608 arbitro= 839 balon= 371  total=10818
SNMOT-123   n_frames= 750  jugador=10681 arbitro= 865 balon= 174  total=11720
SNMOT-124   n_frames= 750  jugador= 7991 arbitro= 815 balon= 391  total=9197
SNMOT-125   n_frames= 750  jugador= 7317 arbitro= 822 balon= 349  total=8488
SNMOT-126   n_frames= 750  jugador= 8743 arbitro= 926 balon= 462  total=10131
SNMOT-130   n_frames= 750  jugador= 9775 arbitro= 787 balon= 261  total=10823
SNMOT-133   n_frames= 750  jugador= 9569 arbitro=1326 balon= 395  total=11290
SNMOT-134   n_frames= 750  jugador= 7098 arbitro=1030 balon= 643  tot

In [4]:
# ---- CELDA 3: TRACKING sobre los 20 clips (acumulado multi-secuencia) ----
import motmetrics as mm

accs=[]; nombres_acc=[]
for nombre in clips_eval:
    tu,gt,clase=cargar_clip(nombre)
    acc=mm.MOTAccumulator(auto_id=True)
    for i,dets in enumerate(tu["frames"]):
        gtf=i+1
        gids,gboxes=[],[]
        for (tid,x,y,w,h) in gt.get(gtf,[]):
            if clase.get(tid)=="persona": gids.append(tid); gboxes.append((x,y,w,h))
        tids,tboxes=[],[]
        for d in dets:
            if d.get("rol") in ("jugador","portero"):
                x1,y1,x2,y2=d["bbox"]; tids.append(d["id"]); tboxes.append((x1,y1,x2-x1,y2-y1))
        if gboxes and tboxes:
            dist=np.zeros((len(gboxes),len(tboxes)))
            for gi,gb in enumerate(gboxes):
                for ti,tb in enumerate(tboxes):
                    v=iou_xywh(gb,tb); dist[gi,ti]=1-v if v>=0.5 else np.nan
        else:
            dist=np.empty((len(gboxes),len(tboxes)))
        acc.update(gids,tids,dist)
    accs.append(acc); nombres_acc.append(nombre)

mh=mm.metrics.create()
resumen=mh.compute_many(accs, names=nombres_acc,
                        metrics=['mota','idf1','num_switches','mostly_tracked','mostly_lost',
                                 'num_false_positives','num_misses','num_objects','num_unique_objects'],
                        generate_overall=True)
print("=== TRACKING — por clip + OVERALL (última fila) ===\n")
import pandas as pd
pd.set_option('display.width',200)
print(resumen[['mota','idf1','num_switches','mostly_tracked','num_unique_objects']].to_string())
print()
ov=resumen.loc['OVERALL']
print(f"OVERALL -> MOTA={ov['mota']:.3f}  IDF1={ov['idf1']:.3f}  ID switches={int(ov['num_switches'])}")


=== TRACKING — por clip + OVERALL (última fila) ===

               mota      idf1  num_switches  mostly_tracked  num_unique_objects
SNMOT-116  0.791548  0.522163           127              18                  24
SNMOT-117  0.854327  0.771897            60              22                  27
SNMOT-118  0.840405  0.759604            39              15                  21
SNMOT-119  0.854515  0.874848            37              18                  22
SNMOT-121  0.896772  0.714122            46              21                  21
SNMOT-122  0.834437  0.518428           146              19                  22
SNMOT-123  0.902508  0.618468            76              21                  22
SNMOT-124  0.818439  0.657820            39              18                  22
SNMOT-125  0.800326  0.552207           102              14                  22
SNMOT-126  0.900769  0.709288            28              22                  22
SNMOT-130  0.917887  0.727254            43              19        

In [5]:
# ---- CELDA 4 (corregida): clasificación de equipos con asignación óptima por clip ----
def etq_gt_fina(desc):
    d=desc.lower()
    if "ball" in d: return "balon"
    if "referee" in d: return "arbitro"
    if "goalkeeper" in d: return "portero_izq" if "left" in d else "portero_der"
    if "player" in d: return "jugador_izq" if "left" in d else "jugador_der"
    return "otro"

eq_correcto=eq_total=0
g_ii=g_id=g_dd=g_di=0
acc_por_clip=[]
for nombre in clips_eval:
    tu,gt,clase=cargar_clip(nombre)
    cfg=configparser.ConfigParser(); cfg.read(TEST_DIR/nombre/"gameinfo.ini")
    etq_gt={}
    for key in cfg["Sequence"]:
        if key.startswith("trackletid_"): etq_gt[int(key.split("_")[1])]=etq_gt_fina(cfg["Sequence"][key])
    # contar (equipo_kmeans, lado_gt) sobre jugadores emparejados de este clip
    c={0:{"izq":0,"der":0},1:{"izq":0,"der":0}}
    for i,dets in enumerate(tu["frames"]):
        gtl=[(tid,(x,y,w,h)) for (tid,x,y,w,h) in gt.get(i+1,[])]
        usado=[False]*len(gtl)
        for d in dets:
            if d.get("rol")!="jugador" or d.get("equipo") not in (0,1): continue
            x1,y1,x2,y2=d["bbox"]; tb=(x1,y1,x2-x1,y2-y1)
            mejor,mj=0.5,-1
            for j,(tid,gb) in enumerate(gtl):
                if usado[j]: continue
                v=iou_xywh(tb,gb)
                if v>mejor: mejor,mj=v,j
            if mj>=0:
                usado[mj]=True
                egt=etq_gt.get(gtl[mj][0],"otro")
                if egt=="jugador_izq": c[d["equipo"]]["izq"]+=1
                elif egt=="jugador_der": c[d["equipo"]]["der"]+=1
    # asignación óptima por clip
    A=c[0]["izq"]+c[1]["der"]   # 0->izq, 1->der
    B=c[0]["der"]+c[1]["izq"]   # 0->der, 1->izq
    tot=A+B
    eq_correcto+=max(A,B); eq_total+=tot
    if tot>0: acc_por_clip.append(max(A,B)/tot)
    if A>=B:
        g_ii+=c[0]["izq"]; g_id+=c[1]["izq"]; g_dd+=c[1]["der"]; g_di+=c[0]["der"]
    else:
        g_ii+=c[1]["izq"]; g_id+=c[0]["izq"]; g_dd+=c[0]["der"]; g_di+=c[1]["der"]

acc=np.array(acc_por_clip)
print("=== CLASIFICACIÓN DE EQUIPOS — asignación óptima por clip (20 clips) ===\n")
print(f"Acierto de equipo (micro): {eq_correcto}/{eq_total} = {100*eq_correcto/max(eq_total,1):.1f}%")
print(f"Acierto por clip: media={100*acc.mean():.1f}%  std={100*acc.std():.1f}%  min={100*acc.min():.1f}%")
print()
print("Matriz jugadores alineada por clip (GT filas / predicción columnas):")
print(f"  GT izq -> izq:{g_ii}  der:{g_id}")
print(f"  GT der -> der:{g_dd}  izq:{g_di}")

=== CLASIFICACIÓN DE EQUIPOS — asignación óptima por clip (20 clips) ===

Acierto de equipo (micro): 227739/248622 = 91.6%
Acierto por clip: media=92.0%  std=5.1%  min=78.8%

Matriz jugadores alineada por clip (GT filas / predicción columnas):
  GT izq -> izq:112991  der:12082
  GT der -> der:114748  izq:8801


In [6]:
# ---- DIAGNÓSTICO ÁRBITRO: ¿no se detecta, o se detecta y se etiqueta mal? ----
tot_gt=tot_arb=tot_jug=tot_no=0
print(f"{'clip':11} {'GT_arb':>7} {'->arbitro':>10} {'->jugador':>10} {'no_detect':>10}")
for nombre in clips_eval:
    tu,gt,clase=cargar_clip(nombre)
    n_gt=n_arb=n_jug=n_no=0
    for i,dets in enumerate(tu["frames"]):
        gtarb=[(x,y,w,h) for (tid,x,y,w,h) in gt.get(i+1,[]) if clase.get(tid)=="arbitro"]
        preds=[(d.get("rol"),d["bbox"]) for d in dets if d.get("rol") in ("arbitro","jugador")]
        for gb in gtarb:
            n_gt+=1
            mejor,rol_m=0.5,None
            for rol,bb in preds:
                x1,y1,x2,y2=bb; v=iou_xywh((x1,y1,x2-x1,y2-y1),gb)
                if v>mejor: mejor,rol_m=v,rol
            if rol_m=="arbitro": n_arb+=1
            elif rol_m=="jugador": n_jug+=1
            else: n_no+=1
    print(f"{nombre:11} {n_gt:7} {n_arb:10} {n_jug:10} {n_no:10}")
    tot_gt+=n_gt; tot_arb+=n_arb; tot_jug+=n_jug; tot_no+=n_no
print(f"\n{'TOTAL':11} {tot_gt:7} {tot_arb:10} {tot_jug:10} {tot_no:10}")
print(f"  ->jugador (visto pero mal etiquetado): {100*tot_jug/max(tot_gt,1):.1f}%")
print(f"  no detectado:                          {100*tot_no/max(tot_gt,1):.1f}%")
print(f"  ->arbitro (correcto):                  {100*tot_arb/max(tot_gt,1):.1f}%")

clip         GT_arb  ->arbitro  ->jugador  no_detect
SNMOT-116       585        543          0         42
SNMOT-117       833        774         10         49
SNMOT-118       749        666          1         82
SNMOT-119       688        637          0         51
SNMOT-121       699        646         24         29
SNMOT-122       782        746          0         36
SNMOT-123       871        797          2         72
SNMOT-124       822        660          0        162
SNMOT-125       678        624          4         50
SNMOT-126       992        859          0        133
SNMOT-130       750        747          0          3
SNMOT-133      1403       1213         29        161
SNMOT-134       978        881          2         95
SNMOT-136       680        598          3         79
SNMOT-141       420        363          0         57
SNMOT-143       535        435          0        100
SNMOT-144       957        895          0         62
SNMOT-145       699        542          6     

In [7]:
# ---- BALÓN: crudo vs interpolado en píxeles (sobre los JSON, sin regenerar) ----
# Gate en píxeles = aproximación del gate de 8 m de fase 7 (no hay campo aquí).
MAX_GAP = 50    # frames: no rellenar huecos más largos que esto
MAX_VEL = 100   # px/frame: no interpolar si la velocidad implícita supera esto (anti-teleport)
DIST    = 15    # px: umbral de acierto contra el GT (igual que la detección)

def centro_xywh(b): return (b[0]+b[2]/2, b[1]+b[3]/2)

def balon_crudo(tu):
    pos={}
    for i,fr in enumerate(tu["frames"]):
        for d in fr:
            if d.get("rol")=="balon":
                x1,y1,x2,y2=d["bbox"]; pos[i]=((x1+x2)/2,(y1+y2)/2); break
    return pos

def interpolar(pos):
    out=dict(pos); fs=sorted(pos)
    for a,b in zip(fs, fs[1:]):
        gap=b-a
        if gap<=1 or gap-1>MAX_GAP: continue
        (xa,ya),(xb,yb)=pos[a],pos[b]
        if np.hypot(xb-xa,yb-ya)/gap > MAX_VEL: continue   # salto = teleport
        for k in range(1,gap):
            t=k/gap; out[a+k]=(xa+(xb-xa)*t, ya+(yb-ya)*t)
    return out

def eval_balon(pos, gt, clase, N):
    TP=FP=FN=0
    for i in range(N):
        gtc=[centro_xywh((x,y,w,h)) for (tid,x,y,w,h) in gt.get(i+1,[]) if clase.get(tid)=="balon"]
        usado=[False]*len(gtc)
        if i in pos:
            cx,cy=pos[i]; mejor,mj=DIST,-1
            for j,(gx,gy) in enumerate(gtc):
                if usado[j]: continue
                dd=np.hypot(cx-gx,cy-gy)
                if dd<mejor: mejor,mj=dd,j
            if mj>=0: TP+=1; usado[mj]=True
            else: FP+=1
        FN+=usado.count(False)
    return TP,FP,FN

g_cr={"TP":0,"FP":0,"FN":0,"cov":0}; g_in={"TP":0,"FP":0,"FN":0,"cov":0}; n_tot=0
for nombre in clips_eval:
    tu,gt,clase=cargar_clip(nombre); N=tu["n_frames"]; n_tot+=N
    cr=balon_crudo(tu); inp=interpolar(cr)
    for pos,acc in [(cr,g_cr),(inp,g_in)]:
        TP,FP,FN=eval_balon(pos,gt,clase,N)
        acc["TP"]+=TP; acc["FP"]+=FP; acc["FN"]+=FN; acc["cov"]+=len(pos)

def fmt(g):
    P=g["TP"]/(g["TP"]+g["FP"]) if g["TP"]+g["FP"]>0 else 0
    R=g["TP"]/(g["TP"]+g["FN"]) if g["TP"]+g["FN"]>0 else 0
    return P,R,(2*P*R/(P+R) if P+R>0 else 0)

print(f"=== BALÓN — crudo vs interpolado (gate: MAX_GAP={MAX_GAP}f, MAX_VEL={MAX_VEL}px/f) ===\n")
Pc,Rc,Fc=fmt(g_cr); Pi,Ri,Fi=fmt(g_in)
print(f"CRUDO      : P={Pc:.3f} R={Rc:.3f} F1={Fc:.3f}  cobertura visual={100*g_cr['cov']/n_tot:.1f}%  (TP={g_cr['TP']} FP={g_cr['FP']} FN={g_cr['FN']})")
print(f"INTERPOLADO: P={Pi:.3f} R={Ri:.3f} F1={Fi:.3f}  cobertura visual={100*g_in['cov']/n_tot:.1f}%  (TP={g_in['TP']} FP={g_in['FP']} FN={g_in['FN']})")

=== BALÓN — crudo vs interpolado (gate: MAX_GAP=50f, MAX_VEL=100px/f) ===

CRUDO      : P=0.811 R=0.439 F1=0.569  cobertura visual=50.8%  (TP=8959 FP=2093 FN=11463)
INTERPOLADO: P=0.633 R=0.542 F1=0.584  cobertura visual=80.4%  (TP=11067 FP=6423 FN=9355)


---

## Conclusión — lectura de los resultados obtenidos

Estas cifras validan la **pila de percepción** (detección → tracking → equipos → roles) en espacio de imagen sobre 20 clips no vistos de SoccerNet, de forma independiente de la etapa de calibración/proyección. En conjunto, el sistema es **sólido donde más importa** (detección de jugadores y clasificación de equipos), **competente en árbitro** y queda **limitado por dos frentes conocidos**: la persistencia de identidad en el tracking y la cobertura del balón.

**Detección.** El detector de personas es el punto fuerte y el cimiento del sistema: **F1 = 0.918** (P = 0.936, R = 0.901) en micro-promedio, y muy estable entre clips (R = 0.898 ± 0.055). De cada 100 jugadores reales se localizan ~90, y 94 de cada 100 detecciones son correctas. El **árbitro** sale notablemente bien (**F1 = 0.878**, R = 0.872), aunque con bastante más varianza entre clips (R = 0.856 ± 0.099): hay partidos donde se detecta peor. El **balón** es la clase más difícil en crudo: **R = 0.437** (F1 = 0.556) y muy variable (R = 0.442 ± 0.168), coherente con su tamaño diminuto e intermitencia.

**Tracking.** **MOTA = 0.833** e **IDF1 = 0.660** sobre el acumulado de los 20 clips, con **1.308 ID switches** y **360/419 (85.9%)** trayectos seguidos casi por completo. La lectura clave está en la **brecha MOTA ≫ IDF1**: como MOTA está dominada por la detección (de hecho, 1 − (FN+FP+IDSW)/GT con los conteos de persona reproduce exactamente 0.833), su valor alto confirma que **detectar y seguir frame a frame funciona bien**; el IDF1 más bajo indica que **el cuello de botella es mantener la misma identidad en el tiempo**, no la detección. Esto es esperable en fútbol (jugadores agrupados, oclusiones, camisetas idénticas) y se refleja en la dispersión entre clips: desde SNMOT-127 (MOTA 0.955, IDF1 0.920, 25 switches) hasta SNMOT-142 (MOTA 0.632, IDF1 0.442, 146 switches). Consecuencia práctica: el sistema es fiable para análisis a nivel de **equipo y de posición**, pero el seguimiento **individual fino** (distancia recorrida o mapa de calor por jugador concreto) está limitado por los switches.

**Clasificación de equipos.** Resultado robusto: de los jugadores detectados y emparejados con el GT, el **91.9%** recibe el equipo correcto (micro), con **92.2% ± 4.4%** por clip y un **mínimo de 83.3%** — es decir, ni el peor clip se degrada mucho. Además, los errores son **simétricos** (≈8% en cada sentido: 8076 izq→der frente a 6173 der→izq), señal de que el agrupador de color **no tiene sesgo sistemático** hacia un equipo y solo falla en los casos genuinamente ambiguos (kits parecidos, sombras, porteros).

**Diagnósticos (árbitro y balón).** El desglose del árbitro es muy informativo: de 18.932 árbitros del GT, el **87.1%** se detecta y etiqueta bien, solo el **1.1%** se confunde con jugador y el **11.8%** no se detecta. Es decir, el margen de mejora del árbitro está **enteramente en la detección**, no en la lógica de rol, que funciona. En el balón, la interpolación cumple su función: sube la **cobertura visual del 52.5% al 84.4%** y el recall de 0.437 a 0.555, a costa de bajar la precisión de 0.762 a 0.601. Es un **compromiso explícito continuidad ↔ precisión**: para el tablero táctico (posesión, pases) interesa tener balón en la mayoría de frames, asumiendo algo de ruido posicional.

**Limitaciones y líneas de mejora.** Por orden de impacto: (1) la **identidad en tracking** (IDF1 0.660, 1.308 switches) es la principal limitación para el análisis individual; mejorarla pasa por reforzar el re-ID por apariencia y el cosido de tracklets en cruces/oclusiones. (2) El **recall de árbitro y balón** (88% y 44% respectivamente) es cuestión de detector: más datos de entrenamiento de esas dos clases minoritarias. (3) La **precisión del balón interpolado** podría afinarse con un gate más estricto si se prioriza exactitud sobre cobertura. Conviene recordar que las métricas de equipos y árbitro están **condicionadas a la detección**, por lo que deben leerse junto al recall, y que esta evaluación **no cubre el error de posición en coordenadas de campo** (eso requiere la calibración PnLCalib y ground truth de Metrica Sports).

**Valoración global.** El sistema demuestra una percepción **fiable y reproducible** sobre material no visto: detecta y sigue jugadores con calidad alta, los asigna a su equipo con ~92% de acierto y distingue al árbitro en ~87% de los casos. Las debilidades restantes están **identificadas y acotadas** (identidad y balón), con líneas de mejora concretas, lo que respalda la validez del enfoque para el análisis táctico objetivo del juego.